In [1]:
import pandas as pd

catalogue = pd.read_csv("../data/catalogue_base.csv")
catalogue.head(1)

,tconst,titre_international,titre_affiche,annee_sortie,duree_min,genres,note_moyenne,nombre_votes,original_language,production_countries,production_companies,overview,poster_path,affiche_url,film_francais
0,tt0006864,Intolerance,Intolérance,1916,210,"Drama,History",7.6,18277,en,United States of America,"Triangle Film Corporation, Wark Producing Corp.","The story of a poor young woman, separated by ...",/lJCabs4TMOsKfnl0BmA5Hf16Ezb.jpg,https://image.tmdb.org/t/p/w500/lJCabs4TMOsKfn...,False


In [2]:
# title.principals par morceaux : nos films + acteurs, actrices, réalisateurs
mes_films = set(catalogue["tconst"])
morceaux = []
lien = "https://datasets.imdbws.com/title.principals.tsv.gz"

for morceau in pd.read_csv(lien, sep="\t", na_values="\\N",
                           usecols=["tconst", "ordering", "nconst", "category"],
                           chunksize=2_000_000):
    est_a_moi = morceau["tconst"].isin(mes_films)
    bon_role = morceau["category"].isin(["actor", "actress", "director"])
    morceaux.append(morceau[est_a_moi & bon_role])

principals = pd.concat(morceaux)
print(principals.shape)

(34479, 4)


In [3]:
# name.basics par morceaux : seulement les personnes de nos films
mes_personnes = set(principals["nconst"])
morceaux = []
lien = "https://datasets.imdbws.com/name.basics.tsv.gz"

for morceau in pd.read_csv(lien, sep="\t", na_values="\\N",
                           usecols=["nconst", "primaryName"],
                           chunksize=2_000_000):
    morceaux.append(morceau[morceau["nconst"].isin(mes_personnes)])

noms = pd.concat(morceaux)

# Ajouter les noms aux rôles
principals = pd.merge(principals, noms, on="nconst", how="left")
principals = principals.dropna(subset="primaryName")
principals = principals.drop_duplicates(subset=["tconst", "nconst", "category"])
principals = principals.sort_values(["tconst", "ordering"])

In [4]:
principals.head(5)

,tconst,ordering,nconst,category,primaryName
0,tt0006864,1,nm0001273,actress,Lillian Gish
2,tt0006864,3,nm0366008,actor,Robert Harron
3,tt0006864,4,nm0550615,actress,Mae Marsh
4,tt0006864,5,nm0877548,actor,F.A. Turner
5,tt0006864,6,nm0208659,actor,Sam De Grasse


In [5]:
# Une ligne par film : les X premiers acteurs, et les réalisateurs
acteurs = principals[principals["category"].isin(["actor", "actress"])]
acteurs = acteurs.groupby("tconst").head(3)          # combien d'acteurs principaux ?
acteurs = acteurs.groupby("tconst")["primaryName"].agg(", ".join).reset_index()
acteurs = acteurs.rename(columns={"primaryName": "acteurs"})

realisateurs = principals[principals["category"] == "director"]
realisateurs = realisateurs.groupby("tconst")["primaryName"].agg(", ".join).reset_index()
realisateurs = realisateurs.rename(columns={"primaryName": "realisateurs"})

# Jointure avec le catalogue
catalogue = pd.merge(catalogue, acteurs, on="tconst", how="left")
catalogue = pd.merge(catalogue, realisateurs, on="tconst", how="left")
print(len(catalogue))
print(catalogue[["acteurs", "realisateurs"]].isna().sum())
catalogue[["titre_affiche", "acteurs", "realisateurs"]].head()

3323
acteurs         250
realisateurs      5
dtype: int64


,titre_affiche,acteurs,realisateurs
0,Intolérance,"Lillian Gish, Robert Harron, Mae Marsh",D.W. Griffith
1,Le Cabinet du docteur Caligari,"Werner Krauss, Conrad Veidt, Friedrich Feher",Robert Wiene
2,Le Kid,"Charles Chaplin, Edna Purviance, Jackie Coogan",Charles Chaplin
3,La charrette fantôme,"Victor Sjöström, Hilda Borgström, Tore Svennberg",Victor Sjöström
4,Les Trois Lumières,"Bernhard Goetzke, Lil Dagover, Walter Janssen",Fritz Lang


In [6]:
catalogue["genres"].str.split(",").explode().value_counts()

genres
Drama          2360
Comedy          855
Crime           627
Romance         516
Action          507
Adventure       416
Thriller        359
Biography       356
Documentary     335
Mystery         226
History         217
Animation       199
War             171
Fantasy         160
Music           155
Family          141
Sci-Fi           97
Horror           94
Sport            87
Musical          77
Film-Noir        54
Western          39
News              5
Name: count, dtype: int64

In [7]:
#Création du dictionnaire des emotions

emotions = {
    "rire":       ["Comedy", "Animation", "Family"],
    "frissonner": ["Horror", "Thriller", "Mystery", "Crime", "Film-Noir"],
    "emouvoir":   ["Romance", "Music", "Musical"],
    "evader":     ["Adventure", "Fantasy", "Sci-Fi", "Animation", "Family"],
    "reflechir":  ["Documentary", "Biography", "History", "News"],
    "vibrer":     ["Action", "War", "Sport", "Western"],
}

In [8]:
# Colonne vrai/faux par emotion

for emotion, genres in emotions.items():
    catalogue[f"emo_{emotion}"] = catalogue["genres"].str.contains("|".join(genres), na=False)

In [9]:
# règle speciale pour le genre Drama qui represente 71% u catalogue

drame_seul = catalogue["genres"] == "Drama"
catalogue["emo_emouvoir"] = catalogue["emo_emouvoir"] | drame_seul
print(drame_seul.sum())

318


In [10]:
# Verification

colonnes_emo = [c for c in catalogue.columns if c.startswith("emo_")]
print(catalogue[colonnes_emo].sum())                        # films par émotion
print((catalogue[colonnes_emo].sum(axis=1) == 0).sum())     # films sans émotion

emo_rire          1070
emo_frissonner    1002
emo_emouvoir      1019
emo_evader         727
emo_reflechir      699
emo_vibrer         785
dtype: int64
0


In [11]:
print(catalogue.shape)
print(len(catalogue))

(3323, 23)
3323


In [12]:
# Encodage des genres (catégoriels)
genres_encodes = catalogue["genres"].str.get_dummies(sep=",")

print(genres_encodes.shape)
genres_encodes.head(5)

(3323, 23)


,Action,Adventure,Animation,Biography,Comedy,Crime,Documentary,Drama,Family,Fantasy,...,Music,Musical,Mystery,News,Romance,Sci-Fi,Sport,Thriller,War,Western
0,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,1,0,0
2,0,0,0,0,1,0,0,1,1,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,1,0,1,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,1,0,1,...,0,0,0,0,0,0,0,0,0,0


In [14]:
# Création d'une colonne encodée pour les langues originales
langues_encodees = pd.get_dummies(catalogue["original_language"], prefix="langue", dtype=int)

In [15]:
# Test des corrélation entre les années de sorties et la durée de chaque film
catalogue[["annee_sortie", "duree_min"]].corr()
# Résultat : Les deux colonnes sont faiblement corrélées (0.18) : aucune redondance donc on garde les 2

,annee_sortie,duree_min
annee_sortie,1.000000,0.184043
duree_min,0.184043,1.000000


In [16]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().set_output(transform="pandas")

numeriques_scalees = scaler.fit_transform(catalogue[["annee_sortie", "duree_min"]])

In [17]:
# Assembler X
X = pd.concat([numeriques_scalees, genres_encodes, langues_encodees], axis=1)

X.shape

(3323, 86)

In [18]:
from sklearn.neighbors import NearestNeighbors

neighbors = NearestNeighbors(n_neighbors=6, metric="euclidean")
neighbors.fit(X)

distances, indices = neighbors.kneighbors(X.iloc[[0]])

In [19]:
position_evades = catalogue[catalogue["tconst"] == "tt0111161"].index[0]

distances, indices = neighbors.kneighbors(X.iloc[[position_evades]])

print(distances[0])

[2.98023224e-08 1.02633080e-01 2.34958709e-01 3.67250827e-01
 3.79806224e-01 3.86545530e-01]


In [20]:
catalogue.iloc[indices[0]][["titre_affiche", "genres", "annee_sortie", "original_language"]]

,titre_affiche,genres,annee_sortie,original_language
1226,Les Évadés,Drama,1994,en
1176,Le club de la chance,Drama,1993,en
1294,Sling Blade,Drama,1996,en
1045,Rain Man,Drama,1988,en
1623,24 heures avant la nuit,Drama,2002,en
952,Love Streams - Torrents d'amour,Drama,1984,en


In [21]:
# Sauvegarde
catalogue.to_csv("../catalogue/films.csv", index=False)